In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from os import mkdir, listdir
from os.path import isdir, isfile
import re
from matplotlib.cm import ScalarMappable
import textwrap
import goatools
from matplotlib import font_manager
from goatools.obo_parser import GODag
from pathlib import Path

OBO_FILE = Path("../data/goatools_data/go-basic.obo")
obodag = GODag(str(OBO_FILE))

# GO Enrichment Analysis 

In [ ]:
annotation_eggnog = pd.read_csv("../data/HydraAEP_eggNOG/hydra_aep.emapper.annotations",sep="\t",skiprows=4)
annotation_eggnog.head()

In [ ]:
annotation_pannzer = pd.read_csv("../data/pannzer/HVAEP/GO.out", sep="\t")
annotation_pannzer.head()

In [ ]:
def apply_front_zeros(go_term):
    go_term = str(go_term)
    go_term = go_term.zfill(7)
    return "GO:" + go_term

annotation_pannzer["GO.terms"] = annotation_pannzer.goid.apply(lambda x: apply_front_zeros(x))
annotation_pannzer["GO.terms"] = annotation_pannzer["GO.terms"].dropna()

In [ ]:
annotation_eggnog = annotation_eggnog[annotation_eggnog["GOs"] != "-"].dropna()

In [ ]:
gene_go_dict_eggnog = {"ID":[],"GO.terms":[]}
for gene, gos in zip(annotation_eggnog["#query"],annotation_eggnog["GOs"]):
    go_terms = gos.split(",")
    for go in go_terms:
        go = go.strip()
        gene_go_dict_eggnog["ID"].append(gene.split(".")[-1])
        gene_go_dict_eggnog["GO.terms"].append(go)
        
gene_go_dict_pannzer = {"ID":[],"GO.terms":[]}
for gene, gos in zip(annotation_pannzer["qpid"],annotation_pannzer["GO.terms"]):
    go_terms = gos.split(",")
    for go in go_terms:
        go = go.strip()
        gene_go_dict_pannzer["ID"].append(gene.split(".")[-1])
        gene_go_dict_pannzer["GO.terms"].append(go)

In [ ]:
annotation_df_eggnog = pd.DataFrame(gene_go_dict_eggnog)
annotation_df_pannzer = pd.DataFrame(gene_go_dict_pannzer)

In [ ]:
with open("../data/goatools_data/associations_eggnog.txt","w") as association_file:
    for gene_id, go_term in zip(annotation_df_eggnog["ID"],annotation_df_eggnog["GO.terms"]):
        association_file.write("{}\t{}\n".format(gene_id,go_term))
with open("../data/goatools_data/associations_pannzer.txt","w") as association_file:
    for gene_id, go_term in zip(annotation_df_pannzer["ID"],annotation_df_pannzer["GO.terms"]):
        association_file.write("{}\t{}\n".format(gene_id,go_term))

In [ ]:
with open("../data/goatools_data/associations_eggnog.txt","r") as association_file:
    lines = association_file.readlines()
print("[*] Length of EggNOG association_file: {}".format(len(lines)))

with open("../data/goatools_data/associations_pannzer.txt","r") as association_file:
    lines = association_file.readlines()
print("[*] Length of Pannzer association_file: {}".format(len(lines)))

In [ ]:
annotation_df_goterms_eggnog = annotation_df_eggnog.copy()
# write population file 
with open("../data/goatools_data/population_eggnog.txt","w") as popfile:
    for identifier in annotation_df_goterms_eggnog.ID.unique():
        popfile.write(identifier+"\n")
        
annotation_df_goterms_pannzer = annotation_df_pannzer.copy()
# write population file 
with open("../data/goatools_data/population_pannzer.txt","w") as popfile:
    for identifier in annotation_df_goterms_pannzer.ID.unique():
        popfile.write(identifier+"\n")

In [ ]:
unique_celltypes = pd.read_table("../results/processedData/normalized_mean/unique_celltypes.table", sep=";")

In [ ]:
unique_celltypes["ID"] = unique_celltypes.ID.apply(lambda x: x.split("-")[1] if type(x) == str else x)

In [ ]:
def construct_celltype_sample_data_for_goatools(unique_celltypes:pd.DataFrame, savep:str, population_table:pd.DataFrame)->int:
    for col in unique_celltypes.columns:
        if col != "ID":
            print("[*] Working with {}".format(col))
            id_to_celltype = unique_celltypes[unique_celltypes[col] == 1.0]
            print("\t[*] Number of unique genes for celltype: {}".format(len(id_to_celltype)))
            with open(savep+"sample_"+col+".txt", "w") as samplefile:
                ident = []
                for identifier in id_to_celltype.ID:
                    if identifier in list(population_table.ID):
                        samplefile.write(identifier+"\n")
                        ident.append(identifier)
                print("\t[*] Remaining IDs for GO analysis: {}".format(len(ident)))
    return 0

In [ ]:
pop_eggnog = pd.read_table("../data/goatools_data/population_eggnog.txt", header=None)
pop_eggnog.columns = ["ID"]

pop_pannzer = pd.read_table("../data/goatools_data/population_pannzer.txt", header=None)
pop_pannzer.columns = ["ID"]

In [ ]:
# write sample files
construct_celltype_sample_data_for_goatools(unique_celltypes, "../data/goatools_data/eggnog/", pop_eggnog)
construct_celltype_sample_data_for_goatools(unique_celltypes, "../data/goatools_data/pannzer/", pop_pannzer)

In [ ]:
# conducting GO enrichment analysis with the find_enrichment.py script of GOATOOLS
# errors if sample files are empty e.g. I_ISC
for celltype in unique_celltypes.columns:
    if celltype != "ID":
        print("[*] Working with: {}".format(celltype))
        print("[*] Starting with eggnog data ...")
        outfile = "../results/processedData/goatools_results/eggnog/celltypes/" + celltype + ".tsv"
        samplefile = "../data/goatools_data/eggnog/sample_" + celltype + ".txt"
        !find_enrichment.py $samplefile ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile --obsolete replace > /dev/null
        print("[*] Continuing with Pannzer data ...")
        outfile = "../results/processedData/goatools_results/pannzer/celltypes/" + celltype + ".tsv"
        samplefile = "../data/goatools_data/pannzer/sample_" + celltype + ".txt"
        !find_enrichment.py $samplefile ../data/goatools_data/population_pannzer.txt ../data/goatools_data/associations_pannzer.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile --obsolete replace > /dev/null        
        print("\t[*] DONE")

In [ ]:
def plot_goa(goafile_enriched:pd.DataFrame,savep:str, filename:str):
    print("[*] Producing plot for {}".format(filename))
    goafile_enriched["ratio_stud"] = goafile_enriched.ratio_in_study.apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
    goafile_enriched["ratio_pop"] = goafile_enriched.ratio_in_pop.apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
    goafile_enriched["amount_in_pop"] = goafile_enriched.ratio_in_pop.apply(lambda x: int(x.split("/")[0]))
    
    categorical_indices = []
    categories = []
    for index, cat in enumerate(list(goafile_enriched.name)):
        if len(cat) >= 30:
            cat = textwrap.fill(cat, width=30)
            categorical_indices.append(index)
        categories.append(cat)
        
    values = list(goafile_enriched.study_count)
    scatter_values = np.array(goafile_enriched.study_count) / np.array(goafile_enriched.amount_in_pop)
    
    pcolors = goafile_enriched.p_fdr_bh
    norm_p_values = np.array(pcolors) / max(pcolors)
    colors=plt.cm.RdBu_r(norm_p_values)
    
    
    # Create figure and axes
    if len(goafile_enriched) == 30:
        fsize = (20,18)
    elif len(goafile_enriched) >= 15:
        fsize = (16,12)
    else:
        fsize = (12,8)
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 16), sharey=True)

    # Plot horizontal bar plot on ax1
    ax1.barh(categories, values, color=colors, edgecolor="black")

    ax1.set_xlabel('Count', fontsize=15, labelpad=20)
    ax1.tick_params(axis='y', labelsize=15) 
    ax1.tick_params(axis='x', labelsize=15) 
    
    
    # Accessing tick labels
    tick_labels = ax1.get_yticklabels()

    # Specify indices of labels to make bold (e.g., 1 and 3 in this example)

    # Update tick labels with LaTeX formatting for bold
    for i, label in enumerate(tick_labels):
        if i in categorical_indices:
            #label.set_fontweight('bold')
            label.set_fontsize(10)  # Optional: Adjust font size if needed
            #label.set_color('blue')  # Optional: Adjust font color if needed
            # Use LaTeX for bold formatting
            #label.set_text(r'\textbf{' + label.get_text() + r'}')
    
    
    ax2.scatter(scatter_values, categories, c=colors, cmap='RdBu_r', 
                label='Gene Ratio (compared to Study)', s=list(goafile_enriched.ratio_stud*1000),edgecolor="black")

    ax2.set_xlabel('Count in Study / Count in Pop', fontsize=15, labelpad=20)
    
    ax2.tick_params(axis='x', labelsize=15) 
    #ax1.set_ylabel('GO Categories')
    ax1.invert_yaxis()
    plt.subplots_adjust(left=0.2, wspace=0.1)
    cbar = fig.colorbar(ScalarMappable(cmap='RdBu_r'), ax=[ax1, ax2], pad = 0.005)
    cbar.set_label('p-values',fontsize=15, labelpad=20)
    cbar.set_ticks([min(norm_p_values), max(norm_p_values)])
    cbar.set_ticklabels([f'{min(goafile_enriched.p_fdr_bh):.4f}', f'{max(goafile_enriched.p_fdr_bh):.4f}'])
    cbar.ax.tick_params(labelsize=12)

    cbar.ax.set_position([0.85, 0.15, 0.03, 0.7])
    
    #plt.show()
    plt.savefig(savep + filename + ".jpg", dpi=400)
    plt.close()
    print("[*] DONE")

In [ ]:
savep_eggnog = "../results/figures/goatools_results/eggnog/celltypes/"
savep_pannzer = "../results/figures/goatools_results/pannzer/celltypes/"
for celltype in unique_celltypes.columns:
    if celltype != "ID":
        filepath_eggnog = "../results/processedData/goatools_results/eggnog/celltypes/" + celltype + ".tsv"
        filepath_pannzer = "../results/processedData/goatools_results/pannzer/celltypes/" + celltype + ".tsv"
        try:
            if isfile(filepath_eggnog):
                goafile = pd.read_table(filepath_eggnog)
                goafile_enriched = goafile[goafile.enrichment == "e"]
                if len(goafile_enriched) > 50:
                    bp_goafile_enriched = goafile_enriched[goafile_enriched.NS == "BP"]
                    cc_goafile_enriched = goafile_enriched[goafile_enriched.NS == "CC"]
                    mf_goafile_enriched = goafile_enriched[goafile_enriched.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_categoriy = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_categoriy):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep=savep_eggnog + process + "_", filename=celltype)
                else:
                    plot_goa(goafile_enriched, savep=savep_eggnog, filename=celltype)
            else:
                print("[*] No eggnog data for: {}".format(celltype))
            if isfile(filepath_pannzer):
                goafile = pd.read_table(filepath_pannzer)
                goafile_enriched = goafile[goafile.enrichment == "e"]
                if len(goafile_enriched) > 50:
                    bp_goafile_enriched = goafile_enriched[goafile_enriched.NS == "BP"]
                    cc_goafile_enriched = goafile_enriched[goafile_enriched.NS == "CC"]
                    mf_goafile_enriched = goafile_enriched[goafile_enriched.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_categoriy = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_categoriy):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep=savep_pannzer + process + "_", filename=celltype)
                else:
                    plot_goa(goafile_enriched, savep=savep_pannzer, filename=celltype)
            else:
                print("[*] No pannzer data for: {}".format(celltype))
        except Exception as e:
            print("[-] ERROR {}".format(e))

            continue

In [ ]:
def construct_study_files_for_transcriptomics(log2FoldChange_df:pd.DataFrame,population_table:pd.DataFrame,
                                              savep:str,result_path:str,experiment:str,annot_type:str)->tuple:
    log2FoldChange_df["ID"] = np.array(log2FoldChange_df.index)
    log2FoldChange_df["ID"] = log2FoldChange_df["ID"].apply(lambda x: x.split(".")[1])
    
    log2FoldChange_df = log2FoldChange_df[log2FoldChange_df["ID"].isin(population_table["ID"])]
    log2FoldChange_df = log2FoldChange_df[log2FoldChange_df["padj"] <= 0.05]
    log2FoldChange_df_up = log2FoldChange_df[log2FoldChange_df["log2FoldChange"] >= 1]
    log2FoldChange_df_down = log2FoldChange_df[log2FoldChange_df["log2FoldChange"] <= -1]
    
    
    print("\t[*] Number of diff genes {}".format(len(log2FoldChange_df)))
    samplefile_up = savep+"sample_up_"+experiment+".txt"
    outfile_up = result_path + "sample_up_"+experiment+".tsv"
    
    
    with open(samplefile_up, "w") as samplefile:
        ident = []
        for identifier in list(log2FoldChange_df_up.ID):
            samplefile.write(identifier+"\n")
            ident.append(identifier)
        print("\t[*] Remaining IDs for Upregulated Genes GO analysis: {}".format(len(ident)))
    
    samplefile_down = savep+"sample_down_"+experiment+".txt"
    outfile_down = result_path + "sample_down_"+experiment+".tsv"
    with open(samplefile_down, "w") as samplefile:
        ident = []
        for identifier in list(log2FoldChange_df_down.ID):
            samplefile.write(identifier+"\n")
            ident.append(identifier)
    print("\t[*] Remaining IDs for Downregulated Genes GO analysis: {}".format(len(ident)))
    
    print("\t[*] Trying to perform find_enrichment analysis ...")
    if annot_type == "eggnog":

        try:
            print("\t[*] Analysing downregulated genes")
            !find_enrichment.py $samplefile_down ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile_down --obsolete replace > /dev/null
        except Exception as e:
            print("\t[**] Warning: {}".format(e))
        try:
            print("\t[*] Analyzing upregulated genes")
            !find_enrichment.py $samplefile_up ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile_up --obsolete replace > /dev/null
        except Exception as e:
            print("\t[**] Warning: {}".format(e))
    
    elif annot_type == "pannzer":
        try:
            print("\t[*] Analysing downregulated genes")
            !find_enrichment.py $samplefile_down ../data/goatools_data/population_pannzer.txt ../data/goatools_data/associations_pannzer.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile_down --obsolete replace > /dev/null
        except Exception as e:
            print("\t[**] Warning: {}".format(e))
        try:
            print("\t[*] Analyzing upregulated genes")
            !find_enrichment.py $samplefile_up ../data/goatools_data/population_pannzer.txt ../data/goatools_data/associations_pannzer.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile_up --obsolete replace > /dev/null
        except Exception as e:
            print("\t[**] Warning: {}".format(e))
            
    else:
        raise Exception("[-] ERROR annot_type is neither eggnog nor pannzer ...")
    
    return outfile_up, outfile_down

In [ ]:
pop_eggnog = pd.read_table("../data/goatools_data/population_eggnog.txt", header=None)
pop_eggnog.columns = ["ID"]
pop_eggnog.head()

In [ ]:
pop_pannzer = pd.read_table("../data/goatools_data/population_pannzer.txt", header=None)
pop_pannzer.columns = ["ID"]
pop_pannzer.head()

In [ ]:
def keep_highest_input_gos(enrichment_df, godag):
    """Keep GO terms that have no ancestor elsewhere in the input frame."""
    go_terms = set(enrichment_df["# GO"])
    highest = go_terms.copy()

    for go_id in go_terms:
        if go_id in godag:
            highest.difference_update(godag[go_id].get_all_parents().intersection(go_terms))

    return enrichment_df[enrichment_df["# GO"].isin(highest)].copy()

In [ ]:
filepath = "../results/deseq2_rsem/tables/"
experiments = listdir(filepath)
for exp in experiments:
    
    if "_vs_Wild_results" not in exp:
    
        print("[*] Working on {}".format(exp))
        if "HydraRecolonization_Conventionalized" in exp or "HydraRecolonization_Cvbct" in exp:
            experiment_name = exp.split("_results")[0]
        else:
            experiment_name = exp.split("_vs_")[0]
        log2FoldChange_df = pd.read_csv(filepath + exp, header=0, index_col=0)
        outfile_up_eggnog, outfile_down_eggnog = construct_study_files_for_transcriptomics(log2FoldChange_df,pop_eggnog,"../data/goatools_data/eggnog/","../results/processedData/goatools_results/eggnog/deseq2_table_enrichment/",experiment_name,annot_type="eggnog")
        outfile_up_pannzer, outfile_down_pannzer = construct_study_files_for_transcriptomics(log2FoldChange_df,pop_pannzer,"../data/goatools_data/pannzer/","../results/processedData/goatools_results/pannzer/deseq2_table_enrichment/",experiment_name, annot_type="pannzer")

        try:
            if isfile(outfile_up_eggnog):
                enriched_up = pd.read_table(outfile_up_eggnog)
                enriched_up = enriched_up[enriched_up.enrichment == "e"]
                enriched_up_high = keep_highest_input_gos(enriched_up,godag=obodag)
                
                
                if len(enriched_up) > 50:
                    bp_goafile_enriched = enriched_up[enriched_up.NS == "BP"]
                    cc_goafile_enriched = enriched_up[enriched_up.NS == "CC"]
                    mf_goafile_enriched = enriched_up[enriched_up.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_categoriy = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_categoriy):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment/" + process + "_", filename=experiment_name+"up")
                else:
                    plot_goa(enriched_up, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment/", filename=experiment_name+"up")
            
                if len(enriched_up_high) > 50:
                    bp_goafile_enriched = enriched_up_high[enriched_up_high.NS == "BP"]
                    cc_goafile_enriched = enriched_up_high[enriched_up_high.NS == "CC"]
                    mf_goafile_enriched = enriched_up_high[enriched_up_high.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_categoriy = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_categoriy):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment/" + process + "_", filename=experiment_name+"up_high")
                else:
                    plot_goa(enriched_up_high, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment/", filename=experiment_name+"up_high")
                       
            

            if isfile(outfile_up_pannzer):
                enriched_up = pd.read_table(outfile_up_pannzer)
                enriched_up = enriched_up[enriched_up.enrichment == "e"]
                
                
                enriched_up_high = keep_highest_input_gos(enriched_up,godag=obodag)
                
                
                if len(enriched_up) > 50:
                    bp_goafile_enriched = enriched_up[enriched_up.NS == "BP"]
                    cc_goafile_enriched = enriched_up[enriched_up.NS == "CC"]
                    mf_goafile_enriched = enriched_up[enriched_up.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment/" + process + "_", filename=experiment_name+"up")
                else:
                    plot_goa(enriched_up, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment/", filename=experiment_name+"up")
            
                if len(enriched_up_high) > 50:
                    bp_goafile_enriched = enriched_up_high[enriched_up_high.NS == "BP"]
                    cc_goafile_enriched = enriched_up_high[enriched_up_high.NS == "CC"]
                    mf_goafile_enriched = enriched_up_high[enriched_up_high.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment/" + process + "_", filename=experiment_name+"up_high")
                else:
                    plot_goa(enriched_up_high, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment/", filename=experiment_name+"up_high")
                       
            
            
            else:
                print("[*] No pannzer goatools file for upregulated genes in: {}".format())

            if isfile(outfile_down_eggnog):
                enriched_down = pd.read_table(outfile_down_eggnog)
                enriched_down = enriched_down[enriched_down.enrichment == "e"]
                
                enriched_down_high = keep_highest_input_gos(enriched_down,godag=obodag)
                
                
                if len(enriched_down) > 50:
                    bp_goafile_enriched = enriched_down[enriched_down.NS == "BP"]
                    cc_goafile_enriched = enriched_down[enriched_down.NS == "CC"]
                    mf_goafile_enriched = enriched_down[enriched_down.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment/" + process + "_", filename=experiment_name+"down")
                else:
                    plot_goa(enriched_down, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment/", filename=experiment_name+"down")
           
                if len(enriched_down_high) > 50:
                    bp_goafile_enriched = enriched_down_high[enriched_down_high.NS == "BP"]
                    cc_goafile_enriched = enriched_down_high[enriched_down_high.NS == "CC"]
                    mf_goafile_enriched = enriched_down_high[enriched_down_high.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment/" + process + "_", filename=experiment_name+"down_high")
                else:
                    plot_goa(enriched_down_high, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment/", filename=experiment_name+"down_high")        
            
            else:
                print("[*] No eggnog goatools file for downregulated genes in: {}".format())

            if isfile(outfile_down_pannzer):
                enriched_down = pd.read_table(outfile_down_pannzer)
                enriched_down = enriched_down[enriched_down.enrichment == "e"]
                enriched_down_high = keep_highest_input_gos(enriched_down,godag=obodag)
                
                
                if len(enriched_down) > 50:
                    bp_goafile_enriched = enriched_down[enriched_down.NS == "BP"]
                    cc_goafile_enriched = enriched_down[enriched_down.NS == "CC"]
                    mf_goafile_enriched = enriched_down[enriched_down.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment/" + process + "_", filename=experiment_name+"down")
                else:
                    plot_goa(enriched_down, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment/", filename=experiment_name+"down")
           
                if len(enriched_down_high) > 50:
                    bp_goafile_enriched = enriched_down_high[enriched_down_high.NS == "BP"]
                    cc_goafile_enriched = enriched_down_high[enriched_down_high.NS == "CC"]
                    mf_goafile_enriched = enriched_down_high[enriched_down_high.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment/" + process + "_", filename=experiment_name+"down_high")
                else:
                    plot_goa(enriched_down_high, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment/", filename=experiment_name+"down_high")        
            
            else:
                print("[*] No pannzer goatools file for upregulated genes in: {}".format())
        except Exception as e:
            print("[-] ERROR: {}".format(e))
            continue
    else:
        print("[*] Skipping: {}".format(exp))
    

In [ ]:
def construct_study_files_for_transcriptomics_unstrict(log2FoldChange_df:pd.DataFrame,population_table:pd.DataFrame,
                                              savep:str,result_path:str,experiment:str,annot_type:str)->tuple:
    log2FoldChange_df["ID"] = np.array(log2FoldChange_df.index)
    log2FoldChange_df["ID"] = log2FoldChange_df["ID"].apply(lambda x: x.split(".")[1])
    
    log2FoldChange_df = log2FoldChange_df[log2FoldChange_df["ID"].isin(population_table["ID"])]
    log2FoldChange_df = log2FoldChange_df[log2FoldChange_df["padj"] <= 0.05]
    log2FoldChange_df_up = log2FoldChange_df[log2FoldChange_df["log2FoldChange"] > 0.0]
    log2FoldChange_df_down = log2FoldChange_df[log2FoldChange_df["log2FoldChange"] < 0.0]
    
    
    print("\t[*] Number of diff genes {}".format(len(log2FoldChange_df)))
    samplefile_up = savep+"sample_up_"+experiment+".txt"
    outfile_up = result_path + "sample_up_"+experiment+".tsv"
    
    
    with open(samplefile_up, "w") as samplefile:
        ident = []
        for identifier in list(log2FoldChange_df_up.ID):
            samplefile.write(identifier+"\n")
            ident.append(identifier)
        print("\t[*] Remaining IDs for Upregulated Genes GO analysis: {}".format(len(ident)))
    
    samplefile_down = savep+"sample_down_"+experiment+".txt"
    outfile_down = result_path + "sample_down_"+experiment+".tsv"
    with open(samplefile_down, "w") as samplefile:
        ident = []
        for identifier in list(log2FoldChange_df_down.ID):
            samplefile.write(identifier+"\n")
            ident.append(identifier)
    print("\t[*] Remaining IDs for Downregulated Genes GO analysis: {}".format(len(ident)))
    
    print("\t[*] Trying to perform find_enrichment analysis ...")
    
    if annot_type == "eggnog":

        try:
            print("\t[*] Analysing downregulated genes")
            !find_enrichment.py $samplefile_down ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile_down --obsolete replace > /dev/null
        except Exception as e:
            print("\t[**] Warning: {}".format(e))
        try:
            print("\t[*] Analyzing upregulated genes")
            !find_enrichment.py $samplefile_up ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile_up --obsolete replace > /dev/null
        except Exception as e:
            print("\t[**] Warning: {}".format(e))
    
    elif annot_type == "pannzer":
        try:
            print("\t[*] Analysing downregulated genes")
            !find_enrichment.py $samplefile_down ../data/goatools_data/population_pannzer.txt ../data/goatools_data/associations_pannzer.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile_down --obsolete replace > /dev/null
        except Exception as e:
            print("\t[**] Warning: {}".format(e))
        try:
            print("\t[*] Analyzing upregulated genes")
            !find_enrichment.py $samplefile_up ../data/goatools_data/population_pannzer.txt ../data/goatools_data/associations_pannzer.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $outfile_up --obsolete replace > /dev/null
        except Exception as e:
            print("\t[**] Warning: {}".format(e))
            
    else:
        raise Exception("[-] ERROR annot_type is neither eggnog nor pannzer ...")
    return outfile_up, outfile_down

In [ ]:
filepath = "../results/deseq2_rsem/tables/"
experiments = listdir(filepath)
for exp in experiments:
    
    if "_vs_Wild_results" not in exp:
    
        print("[*] Working on {}".format(exp))
        if "HydraRecolonization_Conventionalized" in exp or "HydraRecolonization_Cvbct" in exp:
            experiment_name = exp.split("_results")[0]
        else:
            experiment_name = exp.split("_vs_")[0]
        log2FoldChange_df = pd.read_csv(filepath + exp, header=0, index_col=0)
        outfile_up_eggnog, outfile_down_eggnog = construct_study_files_for_transcriptomics_unstrict(log2FoldChange_df,pop_eggnog,"../data/goatools_data/eggnog/","../results/processedData/goatools_results/eggnog/deseq2_table_enrichment_unstrict/",experiment_name,annot_type="eggnog")
        outfile_up_pannzer, outfile_down_pannzer = construct_study_files_for_transcriptomics_unstrict(log2FoldChange_df,pop_pannzer,"../data/goatools_data/pannzer/","../results/processedData/goatools_results/pannzer/deseq2_table_enrichment_unstrict/",experiment_name, annot_type="pannzer")

        try:
            if isfile(outfile_up_eggnog):
                enriched_up = pd.read_table(outfile_up_eggnog)
                enriched_up = enriched_up[enriched_up.enrichment == "e"]
                enriched_up_high = keep_highest_input_gos(enriched_up,godag=obodag)
                
                
                if len(enriched_up) > 50:
                    bp_goafile_enriched = enriched_up[enriched_up.NS == "BP"]
                    cc_goafile_enriched = enriched_up[enriched_up.NS == "CC"]
                    mf_goafile_enriched = enriched_up[enriched_up.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_categoriy = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_categoriy):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment_unstrict/" + process + "_", filename=experiment_name+"up")
                else:
                    plot_goa(enriched_up, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment_unstrict/", filename=experiment_name+"up")
            
                if len(enriched_up_high) > 50:
                    bp_goafile_enriched = enriched_up_high[enriched_up_high.NS == "BP"]
                    cc_goafile_enriched = enriched_up_high[enriched_up_high.NS == "CC"]
                    mf_goafile_enriched = enriched_up_high[enriched_up_high.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_categoriy = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_categoriy):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment_unstrict/" + process + "_", filename=experiment_name+"up_high")
                else:
                    plot_goa(enriched_up_high, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment_unstrict/", filename=experiment_name+"up_high")
                       
            

            if isfile(outfile_up_pannzer):
                enriched_up = pd.read_table(outfile_up_pannzer)
                enriched_up = enriched_up[enriched_up.enrichment == "e"]
                
                
                enriched_up_high = keep_highest_input_gos(enriched_up,godag=obodag)
                
                
                if len(enriched_up) > 50:
                    bp_goafile_enriched = enriched_up[enriched_up.NS == "BP"]
                    cc_goafile_enriched = enriched_up[enriched_up.NS == "CC"]
                    mf_goafile_enriched = enriched_up[enriched_up.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment_unstrict/" + process + "_", filename=experiment_name+"up")
                else:
                    plot_goa(enriched_up, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment_unstrict/", filename=experiment_name+"up")
            
                if len(enriched_up_high) > 50:
                    bp_goafile_enriched = enriched_up_high[enriched_up_high.NS == "BP"]
                    cc_goafile_enriched = enriched_up_high[enriched_up_high.NS == "CC"]
                    mf_goafile_enriched = enriched_up_high[enriched_up_high.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment_unstrict/" + process + "_", filename=experiment_name+"up_high")
                else:
                    plot_goa(enriched_up_high, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment_unstrict/", filename=experiment_name+"up_high")
                       
            
            
            else:
                print("[*] No pannzer goatools file for upregulated genes in: {}".format())

            if isfile(outfile_down_eggnog):
                enriched_down = pd.read_table(outfile_down_eggnog)
                enriched_down = enriched_down[enriched_down.enrichment == "e"]
                
                enriched_down_high = keep_highest_input_gos(enriched_down,godag=obodag)
                
                
                if len(enriched_down) > 50:
                    bp_goafile_enriched = enriched_down[enriched_down.NS == "BP"]
                    cc_goafile_enriched = enriched_down[enriched_down.NS == "CC"]
                    mf_goafile_enriched = enriched_down[enriched_down.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment_unstrict/" + process + "_", filename=experiment_name+"down")
                else:
                    plot_goa(enriched_down, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment_unstrict/", filename=experiment_name+"down")
           
                if len(enriched_down_high) > 50:
                    bp_goafile_enriched = enriched_down_high[enriched_down_high.NS == "BP"]
                    cc_goafile_enriched = enriched_down_high[enriched_down_high.NS == "CC"]
                    mf_goafile_enriched = enriched_down_high[enriched_down_high.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment_unstrict/" + process + "_", filename=experiment_name+"down_high")
                else:
                    plot_goa(enriched_down_high, savep="../results/figures/goatools_results/eggnog/deseq2_table_enrichment_unstrict/", filename=experiment_name+"down_high")        
            
            else:
                print("[*] No eggnog goatools file for downregulated genes in: {}".format())

            if isfile(outfile_down_pannzer):
                enriched_down = pd.read_table(outfile_down_pannzer)
                enriched_down = enriched_down[enriched_down.enrichment == "e"]
                enriched_down_high = keep_highest_input_gos(enriched_down,godag=obodag)
                
                
                if len(enriched_down) > 50:
                    bp_goafile_enriched = enriched_down[enriched_down.NS == "BP"]
                    cc_goafile_enriched = enriched_down[enriched_down.NS == "CC"]
                    mf_goafile_enriched = enriched_down[enriched_down.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment_unstrict/" + process + "_", filename=experiment_name+"down")
                else:
                    plot_goa(enriched_down, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment_unstrict/", filename=experiment_name+"down")
           
                if len(enriched_down_high) > 50:
                    bp_goafile_enriched = enriched_down_high[enriched_down_high.NS == "BP"]
                    cc_goafile_enriched = enriched_down_high[enriched_down_high.NS == "CC"]
                    mf_goafile_enriched = enriched_down_high[enriched_down_high.NS == "MF"]

                    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
                    go_category = ["BP", "CC", "MF"]
                    for goafile, process in zip(egoafiles, go_category):
                        if len(goafile) > 30:
                            goafile = goafile.nsmallest(30, "p_fdr_bh")
                        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment_unstrict/" + process + "_", filename=experiment_name+"down_high")
                else:
                    plot_goa(enriched_down_high, savep="../results/figures/goatools_results/pannzer/deseq2_table_enrichment_unstrict/", filename=experiment_name+"down_high")        
            
            else:
                print("[*] No pannzer goatools file for upregulated genes in: {}".format())
        except Exception as e:
            print("[-] ERROR: {}".format(e))
            continue
    else:
        print("[*] Skipping: {}".format(exp))
    

In [ ]:
2**0.05

# GO Enrichment with Venn-Diagramm Data

In [ ]:
venn_shared_genes = pd.read_csv("../results/processedData/venn_data/shared_genes.txt")
venn_shared_genes["gene"] = venn_shared_genes["gene"].apply(lambda x: x.split(".")[-1])
venn_file_eggnog = "../results/processedData/goatools_results/eggnog/venn/venn_shared_genes_go_analysis_eggnog.txt"
with open(venn_file_eggnog,"w") as outfile:
    for gene in venn_shared_genes["gene"].unique():
        if gene in annotation_df_goterms_eggnog.ID.unique():
            outfile.write("{}\n".format(gene))
            
venn_file_pannzer = "../results/processedData/goatools_results/pannzer/venn/venn_shared_genes_go_analysis_pannzer.txt"
with open(venn_file_pannzer,"w") as outfile:
    for gene in venn_shared_genes["gene"].unique():
        if gene in annotation_df_goterms_pannzer.ID.unique():
            outfile.write("{}\n".format(gene))

In [ ]:
venn_output_enrichment_eggnog = "../results/processedData/goatools_results/eggnog/venn/venn_enrichment_go_eggnog.txt"
venn_output_enrichment_pannzer = "../results/processedData/goatools_results/pannzer/venn/venn_enrichment_go_pannzer.txt"

In [ ]:
!find_enrichment.py $venn_file_eggnog ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $venn_output_enrichment_eggnog --obsolete replace > /dev/null
!find_enrichment.py $venn_file_pannzer ../data/goatools_data/population_pannzer.txt ../data/goatools_data/associations_pannzer.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $venn_output_enrichment_pannzer --obsolete replace > /dev/null

In [ ]:
def keep_highest_input_gos(enrichment_df, godag):
    """Keep GO terms that have no ancestor elsewhere in the input frame."""
    go_terms = set(enrichment_df["GO"])
    highest = go_terms.copy()

    for go_id in go_terms:
        if go_id in godag:
            highest.difference_update(godag[go_id].get_all_parents().intersection(go_terms))

    return enrichment_df[enrichment_df["GO"].isin(highest)].copy()

# Plotting GO-graph

In [ ]:
from collections import defaultdict
import textwrap
import networkx as nx

from goatools.obo_parser import GODag
from goatools.semantic import TermCounts

In [ ]:
enriched = pd.read_table(venn_output_enrichment_eggnog)
enriched = enriched[enriched.enrichment == "e"]

enriched = enriched[enriched["p_fdr_bh"] <= 0.05]

In [ ]:
association = pd.read_table("../data/goatools_data/associations_eggnog.txt", header=None)
association.columns = ["gene","go_id"]

In [ ]:
association

In [ ]:
enriched[enriched["name"] == "multicellular organism development"]

In [ ]:
enriched_class = enriched[enriched["name"] == "multicellular organism development"]["study_items"].to_list()
enriched_class_items = []
for item in enriched_class[0].split(","):
    enriched_class_items.append(item.strip())

In [ ]:
annotations = association[association["gene"].isin(enriched_class_items)]

In [ ]:
godag_network = GODag(
    OBO_FILE,
    optional_attrs={"relationship"}
)

In [ ]:
gene2gos = (
    annotations
    .groupby("gene")["go_id"]
    .agg(set)
    .to_dict()
)

study_associations = {
    gene: gene2gos.get(gene, set())
    for gene in enriched_class_items
}

In [ ]:
termcounts = TermCounts(
    godag_network,
    study_associations,
    relationships={"part_of"},
    prt=None
)

In [ ]:
ROOT = "GO:0007275"

RELATIONSHIPS = {"part_of"}

def get_children(go_id):
    term = godag_network[go_id]

    # Normal is_a children
    children = set(term.children)

    # Additional relationships, e.g. part_of
    for rel in RELATIONSHIPS:
        children |= term.relationship_rev.get(rel, set())

    return children

In [ ]:
def build_go_subgraph(root, max_depth=2, min_genes=1):
    G = nx.DiGraph()

    queue = [(root, 0)]
    visited = set()

    while queue:
        parent, depth = queue.pop(0)

        if parent in visited:
            continue

        visited.add(parent)

        parent_count = len(
            termcounts.go2genes.get(parent, set())
        )

        G.add_node(
            parent,
            count=parent_count,
            name=godag_network[parent].name
        )

        if depth >= max_depth:
            continue

        for child_obj in get_children(parent):

            child = child_obj.item_id

            child_count = len(
                termcounts.go2genes.get(child, set())
            )

            # Remove branches containing none of your genes
            if child_count < min_genes:
                continue

            G.add_node(
                child,
                count=child_count,
                name=child_obj.name
            )

            G.add_edge(parent, child)

            queue.append((child, depth + 1))

    return G

In [ ]:
G = build_go_subgraph(
    ROOT,
    max_depth=2
)

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import textwrap

from networkx.drawing.nx_pydot import graphviz_layout


fig, ax = plt.subplots(figsize=(18, 12))


# ==================================================
# Create attribute-free graph for Graphviz
# ==================================================

node_to_int = {
    node: i
    for i, node in enumerate(G.nodes())
}

int_to_node = {
    i: node
    for node, i in node_to_int.items()
}

G_layout = nx.DiGraph()

G_layout.add_nodes_from(
    node_to_int.values()
)

G_layout.add_edges_from(
    (node_to_int[u], node_to_int[v])
    for u, v in G.edges()
)


# ==================================================
# Calculate hierarchical layout
# ==================================================

pos_tmp = graphviz_layout(
    G_layout,
    prog="dot"
)

pos = {
    int_to_node[i]: xy
    for i, xy in pos_tmp.items()
}


# ==================================================
# Bubble sizes
# ==================================================

counts = nx.get_node_attributes(
    G,
    "count"
)

max_count = max(counts.values())

node_sizes = [
    500 + 5000 * counts[node] / max_count
    for node in G.nodes()
]


# ==================================================
# Draw
# ==================================================

nx.draw_networkx_edges(
    G,
    pos,
    ax=ax,
    arrows=True,
    arrowsize=15,
    width=1
)

nx.draw_networkx_nodes(
    G,
    pos,
    ax=ax,
    node_size=node_sizes,
    alpha=0.8
)


# ==================================================
# Labels
# ==================================================

labels = {}

for node in G.nodes():

    name = G.nodes[node]["name"]
    count = G.nodes[node]["count"]

    wrapped_name = "\n".join(
        textwrap.wrap(name, width=25)
    )

    labels[node] = (
        f"{wrapped_name}\n"
        f"{node}\n"
        f"n={count}"
    )

nx.draw_networkx_labels(
    G,
    pos,
    labels=labels,
    font_size=8,
    ax=ax
)


# ==================================================
# Finish
# ==================================================

ax.set_title(
    "GO:0007275 – multicellular organism development",
    fontsize=16
)

ax.axis("off")

#plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import networkx as nx

def radial_layout_by_depth(G, root, radius_step=2.5):
    """
    Place root at center and nodes on concentric circles
    according to shortest-path distance from root.
    """

    # shortest path distance from root to each node
    depths = nx.single_source_shortest_path_length(G, root)

    # group nodes by depth
    levels = {}
    for node, depth in depths.items():
        levels.setdefault(depth, []).append(node)

    pos = {root: (0.0, 0.0)}

    for depth in sorted(levels.keys()):
        if depth == 0:
            continue

        nodes = sorted(levels[depth])
        n = len(nodes)

        radius = depth * radius_step

        # equally spaced around the circle
        angles = np.linspace(0, 2 * np.pi, n, endpoint=False)

        for node, angle in zip(nodes, angles):
            x = radius * np.cos(angle)
            y = radius * np.sin(angle)
            pos[node] = (x, y)

    return pos

In [ ]:
import matplotlib.pyplot as plt
import textwrap
import numpy as np

ROOT = "GO:0007275"

fig, ax = plt.subplots(figsize=(16, 16))

# radial / mind-map layout
pos = radial_layout_by_depth(G, ROOT, radius_step=3.0)

# node counts
counts = nx.get_node_attributes(G, "count")
max_count = max(counts.values())

# bubble sizes
node_sizes = [
    800 + 6000 * counts[node] / max_count
    for node in G.nodes()
]

# optional: color by depth
depths = nx.single_source_shortest_path_length(G, ROOT)
node_colors = [depths[node] for node in G.nodes()]

# draw edges
nx.draw_networkx_edges(
    G,
    pos,
    ax=ax,
    arrows=True,
    arrowsize=12,
    width=1.2,
    alpha=0.6
)

# draw nodes
nx.draw_networkx_nodes(
    G,
    pos,
    ax=ax,
    node_size=node_sizes,
    node_color=node_colors,
    cmap=plt.cm.Set3,
    alpha=0.9
)

# labels
labels = {}
for node in G.nodes():
    if node == ROOT:
        labels[node] = "multicellular\norganism\ndevelopment\nn={}".format(counts[node])
    else:
        wrapped_name = "\n".join(textwrap.wrap(G.nodes[node]["name"], width=18))
        labels[node] = f"{wrapped_name}\nn={counts[node]}"

nx.draw_networkx_labels(
    G,
    pos,
    labels=labels,
    font_size=8,
    ax=ax
)

ax.set_title("GO Mind Map: multicellular organism development", fontsize=18)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
enriched = pd.read_table("../results/processedData/goatools_results/eggnog/venn/venn_enrichment_go_eggnog.txt")
enriched = enriched[enriched.enrichment == "e"]

enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
#enriched = enriched.rename(columns={"# GO":"GO"})

In [ ]:
bp_df = enriched[enriched["NS"] == "BP"]
bp_df["GO"] = bp_df["# GO"]
bp_df = keep_highest_input_gos(bp_df,godag=obodag)

In [ ]:
bp_df.name

In [ ]:
categorical_indices = []
categories = []
for index, cat in enumerate(list(bp_df.name)):
    if index == 2:
        cat = "SRP-dependent targeting to membrane"
    if index == 3:
        cat = "mRNA catabolic process"
    if index == 4:
        cat = "protein localization to ER"
    if index == 8:
        cat = "regulation of alternative mRNA splicing"
    #if len(cat) >= 40:
    #    cat = textwrap.fill(cat, width=30)
    #    categorical_indices.append(index)
    categories.append(cat)

values = list(bp_df.study_count)
#scatter_values = np.array(bp_df.study_count) / np.array(bp_df.amount_in_pop)

pcolors = bp_df.p_fdr_bh
norm_p_values = np.array(pcolors) / max(pcolors)
colors=plt.cm.YlGnBu_r(norm_p_values)

fig, (ax1) = plt.subplots(1, 1, figsize=(12, 12), sharey=True)

# Plot horizontal bar plot on ax1
ax1.barh(categories, values, color=colors, edgecolor="black")

ax1.set_xlabel('Number of genes with GO-term', fontsize=16, labelpad=20)
ax1.tick_params(axis='y', labelsize=16) 
ax1.tick_params(axis='x', labelsize=16) 


# Accessing tick labels
tick_labels = ax1.get_yticklabels()

# Specify indices of labels to make bold (e.g., 1 and 3 in this example)

# Update tick labels with LaTeX formatting for bold
for i, label in enumerate(tick_labels):
    if i in categorical_indices:
        label.set_fontsize(9)

ax1.invert_yaxis()
plt.subplots_adjust(left=0.2, wspace=0.1)
cbar = fig.colorbar(
    ScalarMappable(cmap='YlGnBu_r'),
    ax=[ax1],
    pad=0.005
)

cbar.set_label(
    'benjamini-hochberg adjusted p-value',
    fontsize=16,
    labelpad=20
)

# 5 Tick-Positionen zwischen min und max
tick_positions = np.linspace(
    min(norm_p_values),
    max(norm_p_values),
    5
)

# Dazu passende ursprüngliche p-Werte
tick_labels = np.linspace(
    min(bp_df.p_fdr_bh),
    max(bp_df.p_fdr_bh),
    5
)

cbar.set_ticks(tick_positions)
cbar.set_ticklabels([f'{x:.2f}' for x in tick_labels])

# Tick-Label-Größe + Größe der Tick-Striche
cbar.ax.tick_params(
    labelsize=16,
    length=8,
    width=2
)


#plt.show()
plt.savefig("../results/figures/goatools_results/eggnog/venn/shared_genes_goterms.svg", dpi=400)
plt.close()

In [ ]:
enriched = pd.read_table(venn_output_enrichment_eggnog)
enriched = enriched[enriched.enrichment == "e"]

enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
enriched = enriched.rename(columns={"# GO":"GO"})

enriched_high = keep_highest_input_gos(enriched, obodag)

bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
go_categoriy = ["BP", "CC", "MF"]
for goafile, process in zip(egoafiles, go_categoriy):
    if len(goafile) > 30:
        goafile = goafile.nsmallest(30, "p_fdr_bh")
    plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/venn/" + process + "_", filename="venn_data")

In [ ]:
enriched = pd.read_table(venn_output_enrichment_pannzer)
enriched = enriched[enriched.enrichment == "e"]

enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
enriched = enriched.rename(columns={"# GO":"GO"})

enriched_high = keep_highest_input_gos(enriched, obodag)

bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
go_categoriy = ["BP", "CC", "MF"]
for goafile, process in zip(egoafiles, go_categoriy):
    if len(goafile) > 30:
        goafile = goafile.nsmallest(30, "p_fdr_bh")
    plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/venn/" + process + "_", filename="venn_data")

In [ ]:
venn_shared_genes = pd.read_csv("../results/processedData/venn_data/shared_genes_upregulated.txt")
venn_shared_genes["gene"] = venn_shared_genes["gene"].apply(lambda x: x.split(".")[-1])
venn_file_eggnog = "../results/processedData/goatools_results/eggnog/venn/venn_shared_genes_go_analysis_eggnog_upregulated.txt"
with open(venn_file_eggnog,"w") as outfile:
    for gene in venn_shared_genes["gene"].unique():
        if gene in annotation_df_goterms_eggnog.ID.unique():
            outfile.write("{}\n".format(gene))
            
venn_file_pannzer = "../results/processedData/goatools_results/pannzer/venn/venn_shared_genes_go_analysis_pannzer_upregulated.txt"
with open(venn_file_pannzer,"w") as outfile:
    for gene in venn_shared_genes["gene"].unique():
        if gene in annotation_df_goterms_pannzer.ID.unique():
            outfile.write("{}\n".format(gene))

In [ ]:
venn_output_enrichment_eggnog = "../results/processedData/goatools_results/eggnog/venn/venn_enrichment_go_eggnog_upregulated.txt"
venn_output_enrichment_pannzer = "../results/processedData/goatools_results/pannzer/venn/venn_enrichment_go_pannzer_upregulated.txt"

!find_enrichment.py $venn_file_eggnog ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $venn_output_enrichment_eggnog --obsolete replace > /dev/null
!find_enrichment.py $venn_file_pannzer ../data/goatools_data/population_pannzer.txt ../data/goatools_data/associations_pannzer.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $venn_output_enrichment_pannzer --obsolete replace > /dev/null

In [ ]:
try:
    enriched = pd.read_table(venn_output_enrichment_eggnog)
    enriched = enriched[enriched.enrichment == "e"]

    enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
    enriched = enriched.rename(columns={"# GO":"GO"})

    enriched_high = keep_highest_input_gos(enriched, obodag)

    bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
    cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
    mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
    go_categoriy = ["BP", "CC", "MF"]
    for goafile, process in zip(egoafiles, go_categoriy):
        if len(goafile) > 30:
            goafile = goafile.nsmallest(30, "p_fdr_bh")
        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/venn/" + process + "_", filename="venn_data_upregulated")
except Exception as e:
    print("[-] ERROR with exception: {}".format(e))

In [ ]:
try:
    enriched = pd.read_table(venn_output_enrichment_pannzer)
    enriched = enriched[enriched.enrichment == "e"]

    enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
    enriched = enriched.rename(columns={"# GO":"GO"})

    enriched_high = keep_highest_input_gos(enriched, obodag)

    bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
    cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
    mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
    go_categoriy = ["BP", "CC", "MF"]
    for goafile, process in zip(egoafiles, go_categoriy):
        if len(goafile) > 30:
            goafile = goafile.nsmallest(30, "p_fdr_bh")
        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/venn/" + process + "_", filename="venn_data_upregulated")
except Exception as e:
    print("[-] ERROR with exception: {}".format(e))

In [ ]:
venn_shared_genes = pd.read_csv("../results/processedData/venn_data/shared_genes_downregulated.txt")
venn_shared_genes["gene"] = venn_shared_genes["gene"].apply(lambda x: x.split(".")[-1])
venn_file_eggnog = "../results/processedData/goatools_results/eggnog/venn/venn_shared_genes_go_analysis_eggnog_downregulated.txt"
with open(venn_file_eggnog,"w") as outfile:
    for gene in venn_shared_genes["gene"].unique():
        if gene in annotation_df_goterms_eggnog.ID.unique():
            outfile.write("{}\n".format(gene))
            
venn_file_pannzer = "../results/processedData/goatools_results/pannzer/venn/venn_shared_genes_go_analysis_pannzer_downregulated.txt"
with open(venn_file_pannzer,"w") as outfile:
    for gene in venn_shared_genes["gene"].unique():
        if gene in annotation_df_goterms_pannzer.ID.unique():
            outfile.write("{}\n".format(gene))

In [ ]:
venn_output_enrichment_eggnog = "../results/processedData/goatools_results/eggnog/venn/venn_enrichment_go_eggnog_downregulated.txt"
venn_output_enrichment_pannzer = "../results/processedData/goatools_results/pannzer/venn/venn_enrichment_go_pannzer_downregulated.txt"

!find_enrichment.py $venn_file_eggnog ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $venn_output_enrichment_eggnog --obsolete replace > /dev/null
!find_enrichment.py $venn_file_pannzer ../data/goatools_data/population_pannzer.txt ../data/goatools_data/associations_pannzer.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $venn_output_enrichment_pannzer --obsolete replace > /dev/null

In [ ]:
enriched = pd.read_table(venn_output_enrichment_eggnog)
enriched = enriched[enriched.enrichment == "e"]

enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
enriched = enriched.rename(columns={"# GO":"GO"})

enriched_high = keep_highest_input_gos(enriched, obodag)

bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
go_categoriy = ["BP", "CC", "MF"]
for goafile, process in zip(egoafiles, go_categoriy):
    if len(goafile) > 30:
        goafile = goafile.nsmallest(30, "p_fdr_bh")
    plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/venn/" + process + "_", filename="venn_data_downregulated")

In [ ]:
enriched = pd.read_table(venn_output_enrichment_pannzer)
enriched = enriched[enriched.enrichment == "e"]

enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
enriched = enriched.rename(columns={"# GO":"GO"})

enriched_high = keep_highest_input_gos(enriched, obodag)

bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
go_categoriy = ["BP", "CC", "MF"]
for goafile, process in zip(egoafiles, go_categoriy):
    if len(goafile) > 30:
        goafile = goafile.nsmallest(30, "p_fdr_bh")
    plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/venn/" + process + "_", filename="venn_data_downregulated")

In [ ]:
annotation_df_goterms_eggnog

In [ ]:
venn_shared_genes = pd.read_csv("../results/processedData/venn_data/shared_wild_conventionalized.txt")
venn_shared_genes["gene"] = venn_shared_genes["gene"].apply(lambda x: x.split(".")[-1])
venn_file_eggnog = "../results/processedData/goatools_results/eggnog/venn/venn_wild_conv_shared_go_analysis_eggnog_upregulated.txt"
with open(venn_file_eggnog,"w") as outfile:
    for gene in venn_shared_genes["gene"].unique():
        if gene in annotation_df_goterms_eggnog.ID.unique():
            outfile.write("{}\n".format(gene))
            
venn_file_pannzer = "../results/processedData/goatools_results/pannzer/venn/venn_wild_conv_shared_go_analysis_eggnog_upregulated.txt"
with open(venn_file_pannzer,"w") as outfile:
    for gene in venn_shared_genes["gene"].unique():
        if gene in annotation_df_goterms_pannzer.ID.unique():
            outfile.write("{}\n".format(gene))

In [ ]:
venn_output_enrichment_eggnog = "../results/processedData/goatools_results/eggnog/venn/venn_enrichment_go_pannzer_wild_conv_upregulated.txt"
venn_output_enrichment_pannzer = "../results/processedData/goatools_results/pannzer/venn/venn_enrichment_go_pannzer_wild_conv_upregulated.txt"

!find_enrichment.py $venn_file_eggnog ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $venn_output_enrichment_eggnog --obsolete replace > /dev/null
!find_enrichment.py $venn_file_pannzer ../data/goatools_data/population_pannzer.txt ../data/goatools_data/associations_pannzer.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $venn_output_enrichment_pannzer --obsolete replace > /dev/null

In [ ]:
try:
    enriched = pd.read_table(venn_output_enrichment_eggnog)
    enriched = enriched[enriched.enrichment == "e"]

    enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
    enriched = enriched.rename(columns={"# GO":"GO"})

    enriched_high = keep_highest_input_gos(enriched, obodag)

    bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
    cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
    mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
    go_categoriy = ["BP", "CC", "MF"]
    for goafile, process in zip(egoafiles, go_categoriy):
        if len(goafile) > 30:
            goafile = goafile.nsmallest(30, "p_fdr_bh")
        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/venn/" + process + "_", filename="wild_conv_upregulated")

    enriched = pd.read_table(venn_output_enrichment_pannzer)
    enriched = enriched[enriched.enrichment == "e"]

    enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
    enriched = enriched.rename(columns={"# GO":"GO"})

    enriched_high = keep_highest_input_gos(enriched, obodag)

    bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
    cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
    mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
    go_categoriy = ["BP", "CC", "MF"]
    for goafile, process in zip(egoafiles, go_categoriy):
        if len(goafile) > 30:
            goafile = goafile.nsmallest(30, "p_fdr_bh")
        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/venn/" + process + "_", filename="wild_conv_upregulated")
except Exception as e:
    print("[-] Exception: {}".format(e))

In [ ]:
venn_shared_genes = pd.read_csv("../results/processedData/venn_data/shared_wild_conventionalized_cvbt.txt")
venn_shared_genes["gene"] = venn_shared_genes["gene"].apply(lambda x: x.split(".")[-1])
venn_file_eggnog = "../results/processedData/goatools_results/eggnog/venn/venn_wild_conv_cvbt_shared_go_analysis_eggnog_upregulated.txt"
with open(venn_file_eggnog,"w") as outfile:
    for gene in venn_shared_genes["gene"].unique():
        if gene in annotation_df_goterms_eggnog.ID.unique():
            outfile.write("{}\n".format(gene))
            
venn_file_pannzer = "../results/processedData/goatools_results/pannzer/venn/venn_wild_conv_cvbt_shared_go_analysis_eggnog_upregulated.txt"
with open(venn_file_pannzer,"w") as outfile:
    for gene in venn_shared_genes["gene"].unique():
        if gene in annotation_df_goterms_pannzer.ID.unique():
            outfile.write("{}\n".format(gene))

In [ ]:
venn_output_enrichment_eggnog = "../results/processedData/goatools_results/eggnog/venn/venn_enrichment_go_pannzer_wild_conv_cvbt_upregulated.txt"
venn_output_enrichment_pannzer = "../results/processedData/goatools_results/pannzer/venn/venn_enrichment_go_pannzer_wild_conv_cvbt_upregulated.txt"

!find_enrichment.py $venn_file_eggnog ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $venn_output_enrichment_eggnog --obsolete replace > /dev/null
!find_enrichment.py $venn_file_pannzer ../data/goatools_data/population_pannzer.txt ../data/goatools_data/associations_pannzer.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $venn_output_enrichment_pannzer --obsolete replace > /dev/null

In [ ]:
try:
    enriched = pd.read_table(venn_output_enrichment_eggnog)
    enriched = enriched[enriched.enrichment == "e"]

    enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
    enriched = enriched.rename(columns={"# GO":"GO"})

    enriched_high = keep_highest_input_gos(enriched, obodag)

    bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
    cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
    mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
    go_categoriy = ["BP", "CC", "MF"]
    for goafile, process in zip(egoafiles, go_categoriy):
        if len(goafile) > 30:
            goafile = goafile.nsmallest(30, "p_fdr_bh")
        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/venn/" + process + "_", filename="wild_conv_cvbt_upregulated")

    enriched = pd.read_table(venn_output_enrichment_pannzer)
    enriched = enriched[enriched.enrichment == "e"]

    enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
    enriched = enriched.rename(columns={"# GO":"GO"})

    enriched_high = keep_highest_input_gos(enriched, obodag)

    bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
    cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
    mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

    egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
    go_categoriy = ["BP", "CC", "MF"]
    for goafile, process in zip(egoafiles, go_categoriy):
        if len(goafile) > 30:
            goafile = goafile.nsmallest(30, "p_fdr_bh")
        plot_goa(goafile, savep="../results/figures/goatools_results/pannzer/venn/" + process + "_", filename="wild_conv_cvbt_upregulated")
except Exception as e:
    print("[-] Exception: {}".format(e))

# GO Cluster Analysis

In [ ]:
pop_eggnog

In [ ]:
goatools_input = "../results/processedData/goatools_results/cluster1.txt"
with open("../results/hierarchical_clustering/cluster_01_genes.txt","r") as infile:
    with open(goatools_input,"w") as outfile:
        counter = 0
        for line in infile.readlines():
            line = line.split(".")[-1]
            line = line.strip()
            if line in pop_eggnog["ID"].to_list():
                outfile.write(line+"\n")
                counter += 1

print(counter)
cluster1_enrichment_output = "../results/processedData/goatools_results/cluster1_enrichment_output.tsv"
!find_enrichment.py $goatools_input ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $cluster1_enrichment_output --obsolete replace > /dev/null


In [ ]:
enriched

In [ ]:

enriched = pd.read_table(cluster1_enrichment_output)
enriched = enriched[enriched.enrichment == "e"]

enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
#enriched = enriched.rename(columns={"# GO":"GO"})

enriched_high = keep_highest_input_gos(enriched, obodag)

bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
go_categoriy = ["BP", "CC", "MF"]
for goafile, process in zip(egoafiles, go_categoriy):
    if len(goafile) > 30:
        goafile = goafile.nsmallest(30, "p_fdr_bh")
    plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/cluster/" + process + "_", filename="cluster1")


In [ ]:
goatools_input = "../results/processedData/goatools_results/cluster2.txt"
with open("../results/hierarchical_clustering/cluster_02_genes.txt","r") as infile:
    with open(goatools_input,"w") as outfile:
        counter = 0
        for line in infile.readlines():
            line = line.split(".")[-1]
            line = line.strip()
            if line in pop_eggnog["ID"].to_list():
                outfile.write(line+"\n")
                counter += 1
            
cluster2_enrichment_output = "../results/processedData/goatools_results/cluster2_enrichment_output.tsv"
!find_enrichment.py $goatools_input ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $cluster2_enrichment_output --obsolete replace > /dev/null


In [ ]:

enriched = pd.read_table(cluster2_enrichment_output)
enriched = enriched[enriched.enrichment == "e"]

enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
#enriched = enriched.rename(columns={"# GO":"GO"})

enriched_high = keep_highest_input_gos(enriched, obodag)

bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
go_categoriy = ["BP", "CC", "MF"]
try:
    for goafile, process in zip(egoafiles, go_categoriy):
        if len(goafile) > 30:
            goafile = goafile.nsmallest(30, "p_fdr_bh")
        plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/cluster/" + process + "_", filename="cluster2")
except Exception as e:
    print(e)


In [ ]:
import os

In [ ]:
dendogram_nodes = []
for file in os.listdir("../results/hierarchical_clustering/"):
    if "distance_clade" in file and file.endswith(".txt"):
        dendogram_nodes.append("../results/hierarchical_clustering/" + file)

In [ ]:
dendogram_nodes

In [ ]:
OVERWRITE_RESULTS = False
for clade_file in dendogram_nodes:
    print("[+] WORKING ON {}".format(clade_file))
    clade = clade_file.split("/")[-1].split("_")[2]
    goatools_input = "../results/processedData/goatools_results/eggnog/cluster/" + "dendogram_{}_genes.txt".format(clade)
     
    with open(clade_file,"r") as infile:
        with open(goatools_input,"w") as outfile:
            counter = 0
            for line in infile.readlines():
                line = line.split(".")[-1]
                line = line.strip()
                if line in pop_eggnog["ID"].to_list():
                    outfile.write(line+"\n")
                    counter += 1

    cluster_enrichment_output = "../results/processedData/goatools_results/eggnog/cluster/dendogram_{}_enrichment_output.tsv".format(clade)
                    
    if os.path.isfile(cluster_enrichment_output) and OVERWRITE_RESULTS == True:

        !find_enrichment.py $goatools_input ../data/goatools_data/population_eggnog.txt ../data/goatools_data/associations_eggnog.txt --annofmt id2gos --alpha 0.05 --pval 0.05 --obo ../data/goatools_data/go-basic.obo --method fdr_bh --outfile $cluster_enrichment_output --obsolete replace > /dev/null
    
    if os.path.isfile(cluster_enrichment_output):
        try:
            enriched = pd.read_table(cluster_enrichment_output)
            enriched = enriched[enriched.enrichment == "e"]

            enriched = enriched[enriched["p_fdr_bh"] <= 0.05]
            #enriched = enriched.rename(columns={"# GO":"GO"})

            enriched_high = keep_highest_input_gos(enriched, obodag)

            bp_goafile_enriched = enriched_high[enriched_high.NS == "BP"]
            cc_goafile_enriched = enriched_high[enriched_high.NS == "CC"]
            mf_goafile_enriched = enriched_high[enriched_high.NS == "MF"]

            egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
            go_categoriy = ["BP", "CC", "MF"]
            for goafile, process in zip(egoafiles, go_categoriy):
                if len(goafile) > 0:
                    if len(goafile) > 30:
                        goafile = goafile.nsmallest(30, "p_fdr_bh")
                    plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/cluster/" + process + "_dendogram_high_", filename=clade)

            bp_goafile_enriched = enriched[enriched.NS == "BP"]
            cc_goafile_enriched = enriched[enriched.NS == "CC"]
            mf_goafile_enriched = enriched[enriched.NS == "MF"]

            egoafiles = [bp_goafile_enriched, cc_goafile_enriched, mf_goafile_enriched]
            go_categoriy = ["BP", "CC", "MF"]
            for goafile, process in zip(egoafiles, go_categoriy):
                if len(goafile) > 0:
                    if len(goafile) > 30:
                        goafile = goafile.nsmallest(30, "p_fdr_bh")
                    plot_goa(goafile, savep="../results/figures/goatools_results/eggnog/cluster/" + process + "_dendogram_", filename=clade)
        except Exception as e:
            print(e)

# GO Universal Terms Analysis

In [ ]:
TOLERANCE = 3
def find_near_universal(enrichment_df, tolerance=TOLERANCE):
    """Summarize GO terms found in all but at most `tolerance` species."""
    if enrichment_df.empty:
        return pd.DataFrame()

    group_columns = ["GO", "name", "category", "enrichment"]
    n_species_per_category = (
        enrichment_df[["experiment", "category"]]
        .drop_duplicates()
        .groupby("category")["experiment"]
        .nunique()
    )

    go_counts = (
        enrichment_df[group_columns + ["experiment"]]
        .drop_duplicates()
        .groupby(group_columns)["experiment"]
        .nunique()
        .reset_index(name="n_experiment_with_GO")
    )
    
    
    study_background_ratio = []
    for study, background in zip(enrichment_df["ratio_in_study"],enrichment_df["ratio_in_pop"]):
        study_background_ratio.append(int(study.split("/")[0])/int(background.split("/")[0]))
    
    enrichment_df["study_background_ratio"] = study_background_ratio
    
    # Each GO term occurs at most once per species, so these statistics describe
    # variation across the species in which that GO term is significant.
    statistics = (
        enrichment_df.groupby(group_columns)
        .agg(
            adjusted_p_value_mean=("p_fdr_bh", "mean"),
            adjusted_p_value_std=("p_fdr_bh", "std"),
            study_background_ratio_mean=("study_background_ratio", "mean"),
            study_background_ratio_std=("study_background_ratio", "std"),
        )
        .reset_index()
    )

    near_universal = go_counts.merge(statistics, on=group_columns, how="left")
    near_universal["n_experiment_in_category"] = near_universal["category"].map(
        n_species_per_category
    )
    near_universal = near_universal[
        near_universal["n_experiment_with_GO"]
        >= near_universal["n_experiment_in_category"] - tolerance
    ].copy()
    near_universal["fraction"] = (
        near_universal["n_experiment_with_GO"]
        / near_universal["n_experiment_in_category"]
    )

    return near_universal.sort_values(
        ["category", "fraction", "n_experiment_with_GO", "enrichment"],
        ascending=[True, False, False, True],
    )


def save_near_universal(enrichment_df, output_stem, tolerance=TOLERANCE):
    near_universal = find_near_universal(enrichment_df, tolerance=tolerance)
    output_stem = Path(output_stem)
    near_universal.to_csv(output_stem.with_suffix(".tsv"), sep="\t", index=False)
    near_universal.to_excel(output_stem.with_suffix(".xlsx"), index=False)
    return near_universal

import os
def run_dataset_analysis(data_path, category):
    enrichment_frames = []
    for file in os.listdir(data_path):
        experiment = file.split("_")[3]
        enrichment = pd.read_table(data_path + '/' + file)
        enrichment = enrichment[enrichment.enrichment == "e"]
        enrichment["GO"] = enrichment["# GO"]
        enrichment = enrichment.assign(category=category, experiment=experiment)
        enrichment_frames.append(enrichment)
    return pd.concat(enrichment_frames, ignore_index=True)

In [ ]:
DATASETS = [
    {
        "name": "Upregulation",
        "data_path": "../results/processedData/goatools_results/eggnog/deseq2_table_enrichment/near_universal_up/data",
        "result_path": "../results/processedData/goatools_results/eggnog/deseq2_table_enrichment/near_universal_up",
        "filename_suffix": "_upregulation",
        "category": "up",
        "output_name": "transcriptome",
    },
    {
        "name": "Downregulation",
        "data_path": "../results/processedData/goatools_results/eggnog/deseq2_table_enrichment/near_universal_down/data",
        "result_path": "../results/processedData/goatools_results/eggnog/deseq2_table_enrichment/near_universal_down",
        "filename_suffix": "_downregulation",
        "category": "down",
        "output_name": "transcriptome",
    },
]

enrichment_results = {}
near_universal_results = {}

for config in DATASETS:
    enrichment = run_dataset_analysis(
        data_path=config["data_path"],
        category=config["category"],
    )
    enrichment_results[config["name"]] = enrichment
    near_universal_results[config["name"]] = save_near_universal(
        enrichment,
        Path(config["result_path"]) / config["output_name"],
    )

